# 10: Profile & Branding System Testing

This notebook tests the complete Profile/Branding workflow:

1. **Credential Backends** — Check available credential storage
2. **User Profile** — Create and persist user configuration
3. **Client Profile** — Create Hillcrest as a client with branding
4. **1Password Integration** — Get credentials from 1Password
5. **Branded Report** — Generate a PDF with client branding
6. **Profile Persistence** — Verify saved profiles reload correctly

## Prerequisites

- 1Password CLI (`op`) installed and authenticated
- Run on macOS for Keychain support

In [1]:
import os
import sys
from pathlib import Path
from datetime import datetime
from IPython.display import display

import siege_utilities as su
su.configure_shared_logging(level="INFO")

sys.path.insert(0, str(Path.cwd().parent))

print(f"Python: {sys.version}")
print(f"Platform: {sys.platform}")

# --- Output configuration ---
OUTPUT_DIR = Path("output") / "notebook_10"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

Python: 3.11.11 (main, Jan 23 2025, 20:04:51) [GCC 13.3.0]
Platform: linux
Output directory: output/notebook_10


## 1. Check Credential Backends

Check which credential storage backends are available.

In [2]:
from siege_utilities.config import CredentialManager, credential_status

status = credential_status()

print("Credential Backend Status:")
for backend, info in status.items():
    symbol = "OK" if info['available'] else "MISSING"
    print(f"  [{symbol}] {backend}: {info['status']}")
    print(f"         {info['description']}")

[siege_utilities] 2026-03-04 12:00:26,360 INFO: Available credential backends: ['files', 'env', 'prompt', '1password']


[siege_utilities] 2026-03-04 12:00:26,361 INFO: Credential manager initialized with backends: {'files': True, 'env': True, 'prompt': True, '1password': True, 'keychain': False}


[siege_utilities] 2026-03-04 12:00:26,362 INFO: Credential search paths: [PosixPath('/home/dheerajchand/git/siege-analytics/siege_utilities/notebooks/credentials'), PosixPath('/home/dheerajchand/.siege_utilities/credentials')]


Credential Backend Status:
  [OK] env: Always available
         Environment variables
  [OK] 1password: Authenticated (6 accounts)
         1Password CLI
  [MISSING] keychain: Not available
         Apple Keychain (requires macOS)
  [OK] prompt: Available (fallback)
         Interactive prompts


In [3]:
# Initialize credential manager
cred_manager = CredentialManager()

print(f"Backend priority: {cred_manager.backend_priority}")
print(f"Available backends: {[k for k,v in cred_manager.available_backends.items() if v]}")
print(f"Default vault: {cred_manager.default_vault}")

[siege_utilities] 2026-03-04 12:00:26,394 INFO: Available credential backends: ['files', 'env', 'prompt', '1password']


[siege_utilities] 2026-03-04 12:00:26,394 INFO: Credential manager initialized with backends: {'files': True, 'env': True, 'prompt': True, '1password': True, 'keychain': False}


[siege_utilities] 2026-03-04 12:00:26,394 INFO: Credential search paths: [PosixPath('/home/dheerajchand/git/siege-analytics/siege_utilities/notebooks/credentials'), PosixPath('/home/dheerajchand/.siege_utilities/credentials')]


Backend priority: ['files', 'env', '1password', 'keychain', 'prompt']
Available backends: ['files', 'env', 'prompt', '1password']
Default vault: Private


## 2. Create User Profile

Create or load user profile with default preferences.

In [4]:
# Modern Person/Actor imports
from siege_utilities.config import User, Client, BrandingConfig, ReportPreferences, HydraConfigManager

manager = HydraConfigManager()

profiles_dir = Path.home() / ".siege_utilities" / "profiles"
users_dir = profiles_dir / "users"
clients_dir = profiles_dir / "clients"

print(f"Users dir:   {users_dir} (exists: {users_dir.exists()})")
print(f"Clients dir: {clients_dir} (exists: {clients_dir.exists()})")

if users_dir.exists():
    existing_users = list(users_dir.glob("*.yaml"))
    print(f"Existing user profiles: {[f.stem for f in existing_users]}")

if clients_dir.exists():
    existing_clients = list(clients_dir.glob("*.yaml"))
    print(f"Existing client profiles: {[f.stem for f in existing_clients]}")

[siege_utilities] 2026-03-04 12:00:26,401 INFO: Initialized HydraConfigManager with config directory: /home/dheerajchand/git/siege-analytics/siege_utilities/siege_utilities/configs


Users dir:   /home/dheerajchand/.siege_utilities/profiles/users (exists: True)
Clients dir: /home/dheerajchand/.siege_utilities/profiles/clients (exists: True)
Existing user profiles: ['dheeraj']
Existing client profiles: ['HILL']


In [5]:
# Create user using modern User model
user = User(
    person_id="dheeraj",
    name="Dheeraj Chand",
    email="dheeraj@siegeanalytics.com",
    username="dheeraj",
    github_login="dheerajchand",
    role="admin",
    preferred_download_directory=str(Path.home() / "Downloads" / "siege_utilities"),
    default_output_format="pdf",
    preferred_map_style="carto-positron",
    default_color_scheme="viridis",
    default_dpi=300,
    default_figure_size=(10, 8),
    enable_logging=True,
    log_level="INFO"
)

print(f"User created: {user.person_id} ({user.name})")
print(f"  Username: {user.username}")
print(f"  Default output: {user.default_output_format}")

User created: dheeraj (Dheeraj Chand)
  Username: dheeraj
  Default output: pdf


In [6]:
# Save and verify user profile
success = manager.save_user(user, profiles_dir=users_dir)
print(f"User profile saved: {success}")

loaded_user = manager.load_user("dheeraj", profiles_dir=users_dir)
if loaded_user:
    print(f"Verified: loaded user {loaded_user.person_id} ({loaded_user.name})")
else:
    print("WARNING: Could not load saved user profile")

[siege_utilities] 2026-03-04 12:00:26,415 INFO: Saved modern User: dheeraj


User profile saved: True
[siege_utilities] 2026-03-04 12:00:26,419 INFO: Loaded modern User: dheeraj


Verified: loaded user dheeraj (Dheeraj Chand)


## 3. Create Hillcrest Client Profile

Create a complete client profile with branding configuration.

In [7]:
# Create Hillcrest branding config
hillcrest_branding = BrandingConfig(
    primary_color="#1a4d2e",      # Forest green (placeholder)
    secondary_color="#4a7c59",    # Sage green (placeholder)
    accent_color="#f4a261",       # Warm accent (placeholder)
    text_color="#2d3436",
    background_color="#ffffff",
    primary_font="Helvetica",
    secondary_font="Georgia",
    logo_path=None,
    header_height=50, footer_height=25,
    margin_top=25, margin_bottom=25, margin_left=20, margin_right=20,
    title_font_size=28, subtitle_font_size=20, body_font_size=12, caption_font_size=10,
    chart_color_palette="viridis",
    chart_background_color="#ffffff",
    chart_grid_color="#e0e0e0"
)

print(f"Branding: {hillcrest_branding.primary_color} / {hillcrest_branding.secondary_color}")
print(f"Fonts: {hillcrest_branding.primary_font} / {hillcrest_branding.secondary_font}")
print(f"Color scheme: {hillcrest_branding.get_color_scheme()}")

Branding: #1a4d2e / #4a7c59
Fonts: Helvetica / Georgia
Color scheme: {'primary': '#1a4d2e', 'secondary': '#4a7c59', 'accent': '#f4a261', 'text': '#2d3436', 'background': '#ffffff', 'chart_background': '#ffffff', 'chart_grid': '#e0e0e0'}


In [8]:
# Create Hillcrest report preferences
hillcrest_reports = ReportPreferences(
    default_format="pdf",
    include_executive_summary=True,
    chart_style="professional",
    page_size="Letter",
    orientation="landscape",
    include_table_of_contents=True,
    include_page_numbers=True,
    chart_quality="high",
    chart_dpi=300,
    include_chart_titles=True,
    include_chart_legends=True,
    sections=["executive_summary", "methodology", "findings", "recommendations"]
)

print(f"Report: {hillcrest_reports.default_format}, {hillcrest_reports.page_size} {hillcrest_reports.orientation}")
print(f"Sections: {hillcrest_reports.sections}")

Report: ReportFormat.PDF, Letter PageOrientation.LANDSCAPE
Sections: ['executive_summary', 'methodology', 'findings', 'recommendations']


In [9]:
# Hillcrest contact info
hillcrest_email = "contact@hillcrest.example.com"
hillcrest_phone = "(555) 123-4567"
hillcrest_website = "https://www.hillcrest.example.com"

print(f"Email:   {hillcrest_email}")
print(f"Phone:   {hillcrest_phone}")
print(f"Website: {hillcrest_website}")

Email:   contact@hillcrest.example.com
Phone:   (555) 123-4567
Website: https://www.hillcrest.example.com


In [10]:
# Create the complete Hillcrest client
hillcrest = Client(
    person_id="hillcrest",
    name="Hillcrest",
    email=hillcrest_email,
    phone=hillcrest_phone,
    website=hillcrest_website,
    client_code="HILL",
    industry="Political Consulting",
    project_count=0,
    client_status="active",
    branding_config=hillcrest_branding,
    report_preferences=hillcrest_reports,
    notes="Primary client for Siege Analytics"
)

# Bidirectional user-client assignment
hillcrest.assign_user(user.person_id)
hillcrest.set_primary_user(user.person_id)
user.assign_client(hillcrest.client_code)
user.set_primary_client(hillcrest.client_code)

print(f"Client created: {hillcrest.name} ({hillcrest.client_code})")
print(f"  Industry: {hillcrest.industry}")
print(f"  Status: {hillcrest.client_status}")
print(f"  Brand color: {hillcrest.branding_config.primary_color}")
print(f"  Assigned users: {hillcrest.get_assigned_users()}")
print(f"  Primary user: {hillcrest.primary_user}")

Client created: Hillcrest (HILL)
  Industry: Political Consulting
  Status: active
  Brand color: #1a4d2e
  Assigned users: ['dheeraj']
  Primary user: dheeraj


In [11]:
# Save and verify Hillcrest profile
success = manager.save_client(hillcrest, profiles_dir=clients_dir)
print(f"Hillcrest profile saved: {success}")

manager.save_user(user, profiles_dir=users_dir)

loaded_hillcrest = manager.load_client("HILL", profiles_dir=clients_dir)
if loaded_hillcrest:
    print(f"Verified: loaded client {loaded_hillcrest.name}")
    print(f"  Code: {loaded_hillcrest.client_code}")
    print(f"  Primary color: {loaded_hillcrest.branding_config.primary_color}")
    print(f"  Assigned users: {loaded_hillcrest.get_assigned_users()}")
else:
    print("WARNING: Could not load saved client profile")

[siege_utilities] 2026-03-04 12:00:26,451 INFO: Saved modern Client: HILL


Hillcrest profile saved: True
[siege_utilities] 2026-03-04 12:00:26,454 INFO: Saved modern User: dheeraj


[siege_utilities] 2026-03-04 12:00:26,459 INFO: Loaded modern Client: HILL


Verified: loaded client Hillcrest
  Code: HILL
  Primary color: #1a4d2e
  Assigned users: ['dheeraj']


## 4. Test 1Password Integration

Retrieve credentials from 1Password (if available).

In [12]:
from siege_utilities.config import get_google_service_account_from_1password

# Check if 1Password is available
if cred_manager.available_backends.get('1password'):
    print("=== 1Password Available ===")
    
    # List stored credentials with siege-utilities tag
    stored_creds = cred_manager.list_stored_credentials()
    print(f"Found {len(stored_creds)} stored credentials (tagged 'siege-utilities'):")
    for cred in stored_creds:
        print(f"  - {cred.get('service', 'unknown')} ({cred.get('backend', 'unknown')})")
    
    if not stored_creds:
        print("  (No items tagged 'siege-utilities' — tag items in 1Password to include them)")
else:
    print("1Password not available - skipping 1Password tests")

=== 1Password Available ===
[siege_utilities] 2026-03-04 12:00:26,489 INFO: 0 items tagged 'siege-utilities' in 1Password (0 total items in vault). Tag items with 'siege-utilities' to include them.


[siege_utilities] 2026-03-04 12:00:26,490 INFO: Found 0 stored credentials


Found 0 stored credentials (tagged 'siege-utilities'):
  (No items tagged 'siege-utilities' — tag items in 1Password to include them)


In [13]:
# Try to get Google Analytics service account from 1Password
# The item title should match what's stored in 1Password

ga_service_account = None

if cred_manager.available_backends.get('1password'):
    # Try common item titles
    item_titles = [
        "Google Analytics Service Account - Multi-Client Reporter",
        "Google Analytics Service Account",
        "GA4 Service Account",
        "Hillcrest GA Service Account"
    ]
    
    for title in item_titles:
        try:
            print(f"Trying: {title}...")
            ga_service_account = get_google_service_account_from_1password(title)
            if ga_service_account:
                print("Found service account!")
                print(f"   Project: {ga_service_account.get('project_id')}")
                print(f"   Email: {ga_service_account.get('client_email')}")
                break
        except Exception as e:
            print(f"   Not found or error: {str(e)[:50]}")
            continue
    
    if not ga_service_account:
        print("WARNING: No GA service account found in 1Password")
        print("To store one, use:")
        print("  from siege_utilities.config import store_ga_service_account_from_file")
        print("  store_ga_service_account_from_file('/path/to/service-account.json')")
else:
    print("Skipping 1Password credential retrieval (not available)")

Trying: Google Analytics Service Account - Multi-Client Reporter...
[siege_utilities] 2026-03-04 12:00:26,510 ERROR: Failed to get Google service account from 1Password: Command '['op', 'item', 'get', 'Google Analytics Service Account - Multi-Client Reporter', '--field=project_id', '--reveal']' returned non-zero exit status 1.


Trying: Google Analytics Service Account...
[siege_utilities] 2026-03-04 12:00:26,521 ERROR: Failed to get Google service account from 1Password: Command '['op', 'item', 'get', 'Google Analytics Service Account', '--field=project_id', '--reveal']' returned non-zero exit status 1.


Trying: GA4 Service Account...
[siege_utilities] 2026-03-04 12:00:26,533 ERROR: Failed to get Google service account from 1Password: Command '['op', 'item', 'get', 'GA4 Service Account', '--field=project_id', '--reveal']' returned non-zero exit status 1.


Trying: Hillcrest GA Service Account...
[siege_utilities] 2026-03-04 12:00:26,544 ERROR: Failed to get Google service account from 1Password: Command '['op', 'item', 'get', 'Hillcrest GA Service Account', '--field=project_id', '--reveal']' returned non-zero exit status 1.


To store one, use:
  from siege_utilities.config import store_ga_service_account_from_file
  store_ga_service_account_from_file('/path/to/service-account.json')


## 5. Generate Branded PDF Report

Create a sample report using Hillcrest branding.

In [14]:
import pandas as pd
import numpy as np

# Create sample data for the report
np.random.seed(42)

# Sample polling data
poll_data = pd.DataFrame({
    'Candidate': ['Smith', 'Jones', 'Williams', 'Undecided'],
    'Support (%)': [42, 38, 12, 8],
    'Previous (%)': [40, 39, 13, 8],
    'Change': ['+2', '-1', '-1', '0']
})

# Sample demographic breakdown
demo_data = pd.DataFrame({
    'Age Group': ['18-29', '30-44', '45-59', '60+'],
    'Smith': [35, 40, 45, 48],
    'Jones': [45, 42, 35, 32],
    'Williams': [15, 12, 12, 10]
})

print("=== Sample Polling Data ===")
display(poll_data)
print("\n=== Demographic Breakdown ===")
display(demo_data)

=== Sample Polling Data ===


,Candidate,Support (%),Previous (%),Change
0,Smith,42,40,+2
1,Jones,38,39,-1
2,Williams,12,13,-1
3,Undecided,8,8,0



=== Demographic Breakdown ===


,Age Group,Smith,Jones,Williams
0,18-29,35,45,15
1,30-44,40,42,12
2,45-59,45,35,12
3,60+,48,32,10


In [15]:
from siege_utilities.reporting.report_generator import ReportGenerator

# Initialize ReportGenerator with Hillcrest branding — output goes to OUTPUT_DIR
rg = ReportGenerator(
    client_name="Hillcrest",
    output_dir=OUTPUT_DIR
)

print("=== ReportGenerator Initialized ===")
print(f"Client: {rg.client_name}")
print(f"Output directory: {rg.output_dir}")

=== ReportGenerator Initialized ===
Client: Hillcrest
Output directory: output/notebook_10


In [16]:
# Build report content
report_content = {
    'sections': [],
    'metadata': {
        'title': 'Sample Polling Report',
        'subtitle': 'Q1 2026 Survey Results',
        'author': 'Siege Analytics',
        'date': datetime.now().strftime('%B %d, %Y'),
        'client': 'Hillcrest'
    }
}

# Add executive summary
executive_summary = """
This polling report presents findings from a survey of 1,200 likely voters 
conducted January 10-15, 2026. Key findings include:

- Smith leads with 42% support, up 2 points from previous poll
- Jones at 38%, down 1 point
- Williams at 12%, essentially unchanged
- 8% remain undecided with 3 weeks until election

Margin of error: +/-2.8 percentage points (95% confidence)
"""

report_content = rg.add_text_section(
    report_content, 
    'Executive Summary', 
    executive_summary
)

print("Added executive summary")

Added executive summary


In [17]:
# Add polling data table
report_content = rg.add_table_section(
    report_content,
    'Current Poll Results',
    poll_data
)

print("Added polling data table")

Added polling data table


In [18]:
# Add demographic breakdown table
report_content = rg.add_table_section(
    report_content,
    'Support by Age Group',
    demo_data
)

print("Added demographic breakdown table")

Added demographic breakdown table


In [19]:
# Add methodology section
methodology = """
Survey Methodology:

- Sample Size: 1,200 likely voters
- Mode: Mixed (60% cell, 40% landline)
- Field Dates: January 10-15, 2026
- Weighting: Age, gender, education, party registration
- Margin of Error: +/-2.8 percentage points

The survey was conducted by Siege Analytics using live interviewers. 
Respondents were selected using stratified random sampling from 
voter file records.
"""

report_content = rg.add_text_section(
    report_content,
    'Methodology',
    methodology
)

print("Added methodology section")

Added methodology section


In [20]:
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak, HRFlowable
from reportlab.lib.pagesizes import letter, landscape
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import inch
from reportlab.lib.enums import TA_CENTER, TA_LEFT

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_path = OUTPUT_DIR / f"hillcrest_poll_report_{timestamp}.pdf"

# --- Hillcrest branding colors ---
hc_primary = colors.HexColor(hillcrest_branding.primary_color)      # #1a4d2e forest green
hc_secondary = colors.HexColor(hillcrest_branding.secondary_color)   # #4a7c59 sage green
hc_accent = colors.HexColor(hillcrest_branding.accent_color)         # #f4a261 warm accent
hc_text = colors.HexColor(hillcrest_branding.text_color)             # #2d3436
hc_white = colors.white

# --- Branded styles ---
styles = getSampleStyleSheet()
styles.add(ParagraphStyle('HC_Title', parent=styles['Title'],
    fontSize=32, textColor=hc_primary, spaceAfter=4))
styles.add(ParagraphStyle('HC_Subtitle', parent=styles['Normal'],
    fontSize=16, textColor=hc_secondary, alignment=TA_CENTER, spaceBefore=8, spaceAfter=20))
styles.add(ParagraphStyle('HC_Heading', parent=styles['Heading1'],
    fontSize=18, textColor=hc_primary, spaceBefore=12, spaceAfter=8))
styles.add(ParagraphStyle('HC_Body', parent=styles['Normal'],
    fontSize=11, textColor=hc_text, leading=15, spaceAfter=4))
styles.add(ParagraphStyle('HC_Bullet', parent=styles['Normal'],
    fontSize=11, textColor=hc_text, leading=15, leftIndent=20, spaceAfter=2))
styles.add(ParagraphStyle('HC_Meta', parent=styles['Normal'],
    fontSize=10, textColor=colors.HexColor('#666666'), alignment=TA_CENTER))

# --- Build the report ---
story = []

# Title page
story.append(Spacer(1, 2.0 * inch))
story.append(HRFlowable(width='70%', thickness=3, color=hc_primary, spaceAfter=16))
story.append(Paragraph('Sample Polling Report', styles['HC_Title']))
story.append(Paragraph('Q1 2026 Survey Results', styles['HC_Subtitle']))
story.append(HRFlowable(width='70%', thickness=1, color=hc_secondary, spaceBefore=4, spaceAfter=30))
story.append(Paragraph('Prepared for: Hillcrest', styles['HC_Meta']))
story.append(Spacer(1, 6))
story.append(Paragraph('Prepared by: Siege Analytics', styles['HC_Meta']))
story.append(Spacer(1, 6))
story.append(Paragraph(f'Date: {datetime.now().strftime("%B %d, %Y")}', styles['HC_Meta']))

# --- Executive Summary ---
story.append(PageBreak())
story.append(HRFlowable(width='100%', thickness=2, color=hc_primary, spaceAfter=8))
story.append(Paragraph('Executive Summary', styles['HC_Heading']))
story.append(Spacer(1, 8))

story.append(Paragraph(
    'This polling report presents findings from a survey of 1,200 likely voters '
    'conducted January 10-15, 2026. Key findings include:',
    styles['HC_Body']))
story.append(Spacer(1, 6))

for b in [
    'Smith leads with 42% support, up 2 points from previous poll',
    'Jones at 38%, down 1 point',
    'Williams at 12%, essentially unchanged',
    '8% remain undecided with 3 weeks until election',
]:
    story.append(Paragraph(f'&bull; {b}', styles['HC_Bullet']))

story.append(Spacer(1, 8))
story.append(Paragraph(
    'Margin of error: +/-2.8 percentage points (95% confidence)',
    styles['HC_Body']))

# --- Current Poll Results ---
story.append(Spacer(1, 20))
story.append(HRFlowable(width='100%', thickness=2, color=hc_primary, spaceAfter=8))
story.append(Paragraph('Current Poll Results', styles['HC_Heading']))
story.append(Spacer(1, 8))

poll_table_data = [poll_data.columns.tolist()] + poll_data.values.tolist()
poll_table = Table(poll_table_data, colWidths=[1.5*inch, 1.2*inch, 1.2*inch, 1.0*inch])
poll_table.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, 0), hc_primary),
    ('TEXTCOLOR', (0, 0), (-1, 0), hc_white),
    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
    ('FONTSIZE', (0, 0), (-1, 0), 11),
    ('FONTNAME', (0, 1), (-1, -1), 'Helvetica'),
    ('FONTSIZE', (0, 1), (-1, -1), 10),
    ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
    ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
    ('BOTTOMPADDING', (0, 0), (-1, 0), 10),
    ('TOPPADDING', (0, 0), (-1, 0), 8),
    ('BOTTOMPADDING', (0, 1), (-1, -1), 6),
    ('TOPPADDING', (0, 1), (-1, -1), 6),
    ('GRID', (0, 0), (-1, -1), 0.5, colors.HexColor('#cccccc')),
    ('ROWBACKGROUNDS', (0, 1), (-1, -1), [hc_white, colors.HexColor('#f0f7f2')]),
]))
story.append(poll_table)

# --- Support by Age Group ---
story.append(Spacer(1, 20))
story.append(HRFlowable(width='100%', thickness=2, color=hc_primary, spaceAfter=8))
story.append(Paragraph('Support by Age Group', styles['HC_Heading']))
story.append(Spacer(1, 8))

demo_table_data = [demo_data.columns.tolist()] + demo_data.values.tolist()
demo_table = Table(demo_table_data, colWidths=[1.2*inch, 1.0*inch, 1.0*inch, 1.0*inch])
demo_table.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, 0), hc_primary),
    ('TEXTCOLOR', (0, 0), (-1, 0), hc_white),
    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
    ('FONTSIZE', (0, 0), (-1, 0), 11),
    ('FONTNAME', (0, 1), (-1, -1), 'Helvetica'),
    ('FONTSIZE', (0, 1), (-1, -1), 10),
    ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
    ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
    ('BOTTOMPADDING', (0, 0), (-1, 0), 10),
    ('TOPPADDING', (0, 0), (-1, 0), 8),
    ('BOTTOMPADDING', (0, 1), (-1, -1), 6),
    ('TOPPADDING', (0, 1), (-1, -1), 6),
    ('GRID', (0, 0), (-1, -1), 0.5, colors.HexColor('#cccccc')),
    ('ROWBACKGROUNDS', (0, 1), (-1, -1), [hc_white, colors.HexColor('#f0f7f2')]),
]))
story.append(demo_table)

# --- Methodology ---
story.append(PageBreak())
story.append(HRFlowable(width='100%', thickness=2, color=hc_primary, spaceAfter=8))
story.append(Paragraph('Methodology', styles['HC_Heading']))
story.append(Spacer(1, 8))

story.append(Paragraph('Survey Methodology:', styles['HC_Body']))
story.append(Spacer(1, 4))

for b in [
    'Sample Size: 1,200 likely voters',
    'Mode: Mixed (60% cell, 40% landline)',
    'Field Dates: January 10-15, 2026',
    'Weighting: Age, gender, education, party registration',
    'Margin of Error: +/-2.8 percentage points',
]:
    story.append(Paragraph(f'&bull; {b}', styles['HC_Bullet']))

story.append(Spacer(1, 10))
story.append(Paragraph(
    'The survey was conducted by Siege Analytics using live interviewers. '
    'Respondents were selected using stratified random sampling from '
    'voter file records.',
    styles['HC_Body']))

# --- Build PDF ---
doc = SimpleDocTemplate(
    str(output_path), pagesize=letter,
    topMargin=1*inch, bottomMargin=1*inch,
    leftMargin=1*inch, rightMargin=1*inch,
)
doc.build(story)

size_kb = output_path.stat().st_size / 1024
print(f"Branded PDF generated: {output_path}")
print(f"Size: {size_kb:.1f} KB, 3 pages")
print(f"Branding: Hillcrest ({hillcrest_branding.primary_color} / {hillcrest_branding.secondary_color})")

Branded PDF generated: output/notebook_10/hillcrest_poll_report_20260304_120028.pdf
Size: 4.5 KB, 3 pages
Branding: Hillcrest (#1a4d2e / #4a7c59)


## 6. Verify Profile Persistence

Verify that profiles are correctly saved and can be reloaded.

In [21]:
# Reload and verify all profiles using modern API
print("=== Verification: Reload All Profiles ===")

# User profile
loaded_user = manager.load_user("dheeraj", profiles_dir=users_dir)
if loaded_user:
    print(f"[OK] User: {loaded_user.person_id} ({loaded_user.name})")
    print(f"   Username: {loaded_user.username}")
    print(f"   Default format: {loaded_user.default_output_format}")
    print(f"   Assigned clients: {loaded_user.get_assigned_clients()}")
else:
    print("ERROR: User profile not found")

# Client profile
loaded_client = manager.load_client("HILL", profiles_dir=clients_dir)
if loaded_client:
    print(f"[OK] Client: {loaded_client.name}")
    print(f"   Code: {loaded_client.client_code}")
    print(f"   Industry: {loaded_client.industry}")
    print(f"   Primary color: {loaded_client.branding_config.primary_color}")
    print(f"   Primary font: {loaded_client.branding_config.primary_font}")
    print(f"   Assigned users: {loaded_client.get_assigned_users()}")
else:
    print("ERROR: Client profile not found")

=== Verification: Reload All Profiles ===
[siege_utilities] 2026-03-04 12:00:28,480 INFO: Loaded modern User: dheeraj


[OK] User: dheeraj (Dheeraj Chand)
   Username: dheeraj
   Default format: pdf
   Assigned clients: ['HILL']
[siege_utilities] 2026-03-04 12:00:28,486 INFO: Loaded modern Client: HILL


[OK] Client: Hillcrest
   Code: HILL
   Industry: Political Consulting
   Primary color: #1a4d2e
   Primary font: Helvetica
   Assigned users: ['dheeraj']


In [22]:
# Check saved profile files
print("=== Profile Files ===")

profiles_base = Path.home() / ".siege_utilities" / "profiles"

for profile_type in ["users", "clients"]:
    profile_dir = profiles_base / profile_type
    if profile_dir.exists():
        files = list(profile_dir.glob("*.yaml"))
        print(f"{profile_type.title()}:")
        for f in files:
            size = f.stat().st_size
            print(f"  - {f.name} ({size} bytes)")
    else:
        print(f"WARNING: {profile_type.title()}: Directory not found")

=== Profile Files ===
Users:
  - dheeraj.yaml (961 bytes)
Clients:
  - HILL.yaml (1756 bytes)


## Summary

Test results summary.

In [23]:
# Summary of all tests
print("=" * 50)
print("PROFILE & BRANDING SYSTEM TEST RESULTS")
print("=" * 50)

results = {
    "Credential Backends": "1password" in [k for k,v in cred_manager.available_backends.items() if v],
    "User Profile Created": manager.load_user("dheeraj", profiles_dir=users_dir) is not None,
    "Client Profile Created": manager.load_client("HILL", profiles_dir=clients_dir) is not None,
    "1Password Integration": cred_manager.available_backends.get('1password', False),
    "PDF Generation": output_path.exists() if 'output_path' in dir() else False,
}

for test, passed in results.items():
    symbol = "PASS" if passed else "FAIL"
    print(f"  {symbol}  {test}")

passed = sum(results.values())
total = len(results)
print(f"\n{passed}/{total} tests passed.")

if passed == total:
    print("\nAll Profile/Branding tests passed!")
else:
    print("\nSome tests need attention.")

# Clean up HydraConfigManager
manager.cleanup()

PROFILE & BRANDING SYSTEM TEST RESULTS
[siege_utilities] 2026-03-04 12:00:28,502 INFO: Loaded modern User: dheeraj


[siege_utilities] 2026-03-04 12:00:28,508 INFO: Loaded modern Client: HILL


  PASS  Credential Backends
  PASS  User Profile Created
  PASS  Client Profile Created
  PASS  1Password Integration
  PASS  PDF Generation

5/5 tests passed.

All Profile/Branding tests passed!


## Next Steps

If all tests pass:

1. **Update Hillcrest branding colors** - Edit the BrandingConfig with actual Hillcrest colors
2. **Add logo** - Set `logo_path` to actual Hillcrest logo file
3. **Store GA credentials** - If you have GA service account, store it:
   ```python
   from siege_utilities.config import store_ga_service_account_from_file
   store_ga_service_account_from_file('/path/to/service-account.json', 
                                       item_title='Hillcrest GA Service Account')
   ```
4. **Test with real data** - Use actual polling data from Hillcrest projects